# Task 2 re-run — let the models actually finish

In the first run, Attention U-Net and U-Net both hit `best_epoch = 40` out of 40.
Validation loss was **still improving** when the epoch cap stopped them. Only U-Net++
early-stopped properly, at epoch 31. So the three models were not compared at
convergence, and the 0.0004 Jaccard gap between Attention U-Net and U-Net means nothing.

This notebook changes exactly one thing: **`n_epochs` 40 -> 150**. Everything else is
identical — same seed, same split, same crops, same Dice+BCE loss, same patience 5. Now
early stopping decides when to stop, not the cap.

Task 1 and the loss ablation are not repeated; your existing numbers for those are fine.

## How to run

1. **Runtime -> Change runtime type -> T4 GPU -> Save**
2. **Runtime -> Run all**
3. At the "Choose Files" button, upload **`module2_code.zip`** only (16 KB — you do not
   need the data zip, the membrane data downloads itself)
4. Wait ~45 min. The results zip downloads to your Mac at the end.

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
      or "NO GPU — Runtime > Change runtime type > T4 GPU")

## Upload `module2_code.zip`

In [ ]:
import os, shutil, subprocess, sys, zipfile
from google.colab import files

WORK = "/content/work"
os.makedirs(WORK, exist_ok=True)
os.chdir("/content")

for name in files.upload():
    with zipfile.ZipFile(f"/content/{name}") as z:
        z.extractall(WORK)
    print("unpacked", name)

os.chdir(WORK)
sys.path.insert(0, WORK)
if not os.path.isdir(f"{WORK}/src"):
    raise FileNotFoundError("no src/ — that was not module2_code.zip")
print("code:", sorted(os.listdir("src")))

In [ ]:
# Membrane data is public — fetch it here.
if not os.path.isdir("data/train/images"):
    subprocess.run(["git", "clone", "--depth", "1", "-q",
                    "https://github.com/zhixuhao/unet.git", "/tmp/unet_repo"], check=True)
    src = "/tmp/unet_repo/data/membrane"
    for dst, pattern in [("data/train/images", "train/image"),
                         ("data/train/labels", "train/label")]:
        os.makedirs(dst, exist_ok=True)
        for f in os.listdir(f"{src}/{pattern}"):
            shutil.copy(f"{src}/{pattern}/{f}", dst)
    os.makedirs("data/test", exist_ok=True)
    for f in os.listdir(f"{src}/test"):
        if not f.endswith("_predict.png"):
            shutil.copy(f"{src}/test/{f}", "data/test")

print("membrane:", len(os.listdir("data/train/images")), "images,",
      len(os.listdir("data/train/labels")), "labels")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

from src.attention_unet import AttentionUNet, UNet
from src.losses import LOSSES
from src.membrane import load_membrane
from src.results import results_table, save_run
from src.seed import get_device, set_seed
from src.unetpp import UNetPlusPlus
import src.segmentation as seg

device = get_device()
print("device:", device)

## Train

`n_epochs = 150` is a ceiling, not a target. With patience 5 each model stops on its own
once validation loss stops improving. Watch the printed `Early stopping at epoch N` line
— if a model reports epoch 150 again, the cap is still too low and we raise it further.

In [ ]:
MODELS = {
    "attention_unet": lambda: AttentionUNet(base=64),
    "unet":           lambda: UNet(base=64),
    "unetpp":         lambda: UNetPlusPlus(base=64),
}
CFG = dict(crop_size=256, batch_size=4, lr=1e-4, n_epochs=150, patience=5,
           repeats=8, loss="dice_bce", splits=(20, 5, 5), seed=27)

results, stop_info, loaders = {}, {}, {}

for name, factory in MODELS.items():
    set_seed(CFG["seed"])
    print(f"\n{'='*70}\n{name}\n{'='*70}")
    tr, va, te = load_membrane(root="data", crop_size=CFG["crop_size"],
                               batch_size=CFG["batch_size"], splits=CFG["splits"],
                               repeats=CFG["repeats"], seed=CFG["seed"])
    model = factory().to(device)
    criterion = LOSSES[CFG["loss"]]()
    optimizer = torch.optim.Adam(model.parameters(), lr=CFG["lr"])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min",
                                                           factor=0.5, patience=3)
    history = seg.fit(model, tr, va, criterion, optimizer, device,
                      n_epochs=CFG["n_epochs"], patience=CFG["patience"],
                      scheduler=scheduler, checkpoint_path=f"{WORK}/{name}.pt")
    metrics, _ = seg.evaluate(model, te, criterion, device)

    results[name] = metrics
    stop_info[name] = (history["best_epoch"], len(history["train_loss"]))
    save_run(tag=f"{name}_rerun", config={**CFG, "model": name},
             metrics=metrics, history=history, model=model)
    if name == "attention_unet":
        best_model, best_loader = model, te
    print({k: round(v, 4) for k, v in metrics.items()})

## Did they converge this time?

In [ ]:
for name, (best, ran) in stop_info.items():
    ok = "converged" if best < ran else "STILL CAPPED — raise n_epochs"
    print(f"{name:16s} best epoch {best:3d} of {ran:3d} run   -> {ok}")

## Table 2 — paste this into the report

In [ ]:
labels = {"attention_unet": "Attention U-Net", "unet": "U-Net", "unetpp": "U-Net++"}
print(results_table({labels[k]: v for k, v in results.items()},
                    ["jaccard", "f1", "recall", "precision", "accuracy"], "{:.4f}"))

## Qualitative results and attention maps

In [ ]:
best_model.eval()
fig, axes = plt.subplots(3, 3, figsize=(12, 12))
with torch.no_grad():
    for col, (image, mask) in zip(range(3), best_loader):
        prob = torch.sigmoid(best_model(image.to(device)))[0, 0].cpu()
        for row, (img, title) in enumerate([(image[0, 0], "original"),
                                            (mask[0, 0], "ground truth"),
                                            (prob >= 0.5, "prediction")]):
            axes[row][col].imshow(img, cmap="gray")
            axes[row][col].set_title(title); axes[row][col].axis("off")
fig.tight_layout(); fig.savefig("task2_qualitative.png", dpi=150)

In [ ]:
image, _ = next(iter(best_loader))
with torch.no_grad():
    _, attention = best_model(image.to(device), return_attention=True)
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
axes[0].imshow(image[0, 0], cmap="gray"); axes[0].set_title("input")
for ax, a in zip(axes[1:], attention):
    ax.imshow(a[0, 0].cpu(), cmap="inferno"); ax.set_title(f"gate @ {tuple(a.shape[-2:])}")
for ax in axes: ax.axis("off")
fig.tight_layout(); fig.savefig("task2_attention.png", dpi=150)

## Download

In [ ]:
import datetime
out = "/content/out"
shutil.rmtree(out, ignore_errors=True); os.makedirs(out)
shutil.copytree(f"{WORK}/results", f"{out}/results")
for png in [p for p in os.listdir(WORK) if p.endswith(".png")]:
    shutil.copy(f"{WORK}/{png}", out)
stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
files.download(shutil.make_archive(f"/content/task2_rerun_{stamp}", "zip", out))

---
## Reading the result

**If the ranking still comes out flat** (all three within ~0.01 Jaccard) that is the
answer, and it is a perfectly good one to report: on this dataset, neither attention
gates nor nested skips beat plain U-Net. Membranes cover ~22% of every image and are
distributed everywhere, so there is little irrelevant background for an attention gate
to suppress. Attention helps when the target is small and sparse — a polyp in an
otherwise empty frame — which is not this task.

**Do not claim a winner on a 0.001 difference.** The test set is 5 images. To make a
real claim you would run 3 seeds and report mean +- std.